# E0: Small FCN with Cross-Entropy

We train a small FCN as a reference for the U-Net experiments. Each pixel is assigned to one of three classes: foreground, background, or border.

The saved outputs below are from a short smoke run.


In [1]:
from pathlib import Path
import os
import subprocess
import sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Locate the project whether launched from the root or notebooks/.
candidates = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next(
    (p for p in candidates if (p / "configs/e0_small_fcn.json").is_file()),
    None,
)
if REPO_ROOT is None and IN_COLAB:
    REPO_ROOT = Path("/content/data255_project")
    if not REPO_ROOT.exists():
        subprocess.check_call([
            "git", "clone",
            "https://github.com/Ashley96yy/data255_project.git", str(REPO_ROOT),
        ])
if REPO_ROOT is None or not (REPO_ROOT / "configs/e0_small_fcn.json").is_file():
    raise FileNotFoundError("Open this notebook in an up-to-date data255_project checkout.")

if IN_COLAB:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "-r", str(REPO_ROOT / "requirements.txt"),
    ])
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import hashlib
import inspect
import json
import platform
from datetime import datetime, timezone
from uuid import uuid4

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import PIL
import torch
import torchvision
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader

from src.dataset import PetSegmentationDataset
from src.metrics import CLASS_NAMES
from src.training import fit, seed_everything, seed_worker

# CUDA on Colab; CPU is the portable local reference environment.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__, "| torchvision:", torchvision.__version__)
print("Device:", DEVICE)


Python: 3.12.2
PyTorch: 2.10.0 | torchvision: 0.25.0
Device: cpu


## 1. Data and settings

We use `train_val_split_clean.csv`: 2,937 training images and 736 validation images, with seven invalid masks excluded. Images and masks are resized to 256 × 256. Training uses random horizontal flips; validation uses no augmentation. Mask labels are 0 for foreground, 1 for background, and 2 for border.

The configuration in `configs/e0_small_fcn.json` uses Adam with a learning rate of 0.001, batch size 8, 20 epochs, and seed 42. The test set is reserved for final evaluation.

Set `DOWNLOAD_DATA = True` if the data have not been downloaded. To use data stored elsewhere, set `E0_DATA_ROOT` to the folder containing `oxford-iiit-pet/`.


In [2]:
RUN_MODE = "smoke"  # change to "full" for the full experiment
DOWNLOAD_DATA = False  # set True to download into the selected data root

if RUN_MODE not in {"smoke", "full"}:
    raise ValueError("RUN_MODE must be 'smoke' or 'full'.")
CONFIG = json.loads((REPO_ROOT / "configs/e0_small_fcn.json").read_text())
CONFIG.update(
    mode=RUN_MODE,
    epochs=1 if RUN_MODE == "smoke" else CONFIG["epochs"],
    max_train_batches=2 if RUN_MODE == "smoke" else None,
    max_val_batches=2 if RUN_MODE == "smoke" else None,
)
SPLIT_CSV = REPO_ROOT / CONFIG["split_csv"]
split_df = pd.read_csv(SPLIT_CSV)
assert not split_df["dataset_index"].duplicated().any()
split_indices = {
    split: set(split_df.loc[split_df["split"] == split, "dataset_index"])
    for split in ("train", "val", "excluded")
}
assert split_indices["train"].isdisjoint(split_indices["val"])
assert split_indices["excluded"].isdisjoint(
    split_indices["train"] | split_indices["val"]
)
print("Clean split counts:", split_df["split"].value_counts().to_dict())
print("Run mode:", CONFIG["mode"], "| epochs:", CONFIG["epochs"])

data_candidates = [REPO_ROOT / "data", REPO_ROOT / "notebooks/data"]
existing_root = next(
    (p for p in data_candidates if (p / "oxford-iiit-pet/images").is_dir()),
    REPO_ROOT / "data",
)
DATA_ROOT = Path(os.environ.get("E0_DATA_ROOT", str(existing_root))).expanduser()
if not DOWNLOAD_DATA and not (DATA_ROOT / "oxford-iiit-pet/images").is_dir():
    raise FileNotFoundError(
        "Dataset missing. Set DOWNLOAD_DATA=True or E0_DATA_ROOT to the directory "
        "containing oxford-iiit-pet/."
    )

train_dataset = PetSegmentationDataset(
    DATA_ROOT, split="train", split_csv=SPLIT_CSV,
    image_size=CONFIG["image_size"], augment=CONFIG["augment"],
    download=DOWNLOAD_DATA,
)
val_dataset = PetSegmentationDataset(
    DATA_ROOT, split="val", split_csv=SPLIT_CSV,
    image_size=CONFIG["image_size"], augment=False, download=False,
)
assert len(train_dataset) == len(split_indices["train"]) > 0
assert len(val_dataset) == len(split_indices["val"]) > 0
print("Loaded train:", len(train_dataset), "| validation:", len(val_dataset))


Clean split counts: {'train': 2937, 'val': 736, 'excluded': 7}
Run mode: smoke | epochs: 1
Loaded train: 2937 | validation: 736


## 2. Model

The FCN has three convolutional blocks with 32, 64, and 128 channels. Each block contains two 3 × 3 convolutions, batch normalization, and ReLU.

`ConvBlock(3,32) → MaxPool → ConvBlock(32,64) → MaxPool → ConvBlock(64,128) → 1×1 Conv → bilinear upsampling`

The final convolution produces three class logits. Bilinear interpolation brings them back to the input size. This model has no skip connections.


In [3]:
class ConvBlock(nn.Sequential):
    """Two 3×3 convolutions with batch normalization and ReLU."""

    def __init__(self, in_channels, out_channels):
        super().__init__(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )


class SmallFCN(nn.Module):
    """Small FCN that returns logits at the input spatial resolution."""

    def __init__(self, in_channels=3, num_classes=3):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(in_channels, 32),
            nn.MaxPool2d(2),
            ConvBlock(32, 64),
            nn.MaxPool2d(2),
            ConvBlock(64, 128),
        )
        self.classifier = nn.Conv2d(128, num_classes, kernel_size=1)

    def forward(self, inputs):
        input_size = inputs.shape[-2:]
        logits = self.classifier(self.features(inputs))
        return F.interpolate(
            logits, size=input_size, mode="bilinear", align_corners=False
        )


def count_trainable_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

## 3. Forward pass

We check that a random batch of shape `[8, 3, 256, 256]` produces logits of the same shape and a finite Cross-Entropy loss. The cell also checks that the notebook model and `src/models/small_fcn.py` give the same output when using the same weights.


In [4]:
torch.manual_seed(255)
smoke_model = SmallFCN()
images = torch.randn(8, 3, 256, 256)
masks = torch.randint(0, 3, (8, 256, 256), dtype=torch.long)
logits = smoke_model(images)
loss = nn.CrossEntropyLoss()(logits, masks)
assert logits.shape == images.shape
assert torch.isfinite(logits).all() and torch.isfinite(loss)
print("Input:", tuple(images.shape), "| logits:", tuple(logits.shape))
print(f"Synthetic CrossEntropyLoss: {loss.item():.6f}")
print(f"Trainable parameters: {count_trainable_parameters(smoke_model):,}")

from src.models.small_fcn import SmallFCN as ReusableSmallFCN

reference_model = ReusableSmallFCN()
reference_model.load_state_dict(smoke_model.state_dict())
smoke_model.eval()
reference_model.eval()
with torch.no_grad():
    torch.testing.assert_close(smoke_model(images), reference_model(images))
print("Notebook / reusable model parity: passed")
del smoke_model, reference_model, images, masks, logits, loss


Input: (8, 3, 256, 256) | logits: (8, 3, 256, 256)
Synthetic CrossEntropyLoss: 1.122697
Trainable parameters: 287,843


Notebook / reusable model parity: passed


## 4. Training

We train with Cross-Entropy and save the checkpoint with the highest validation mIoU. Validation runs in evaluation mode without gradients.

IoU is calculated from the accumulated confusion matrix over the validation set. The mean includes foreground, background, and border. A class with zero union is excluded from the mean.

The training functions are defined in `src/training.py` and shown below.


In [5]:
from src import training

print(inspect.getsource(training.run_epoch))
print(inspect.getsource(training.fit))


def run_epoch(model, loader, device, optimizer=None, max_batches=None):
    """Run one train or validation pass and aggregate all observed pixels."""
    if max_batches is not None and max_batches < 1:
        raise ValueError("max_batches must be positive or None.")
    training = optimizer is not None
    model.train(training)
    criterion = nn.CrossEntropyLoss()
    matrix = torch.zeros(3, 3, dtype=torch.int64)
    total_loss = 0.0
    pixels = samples = batches = 0
    iterator = loader if max_batches is None else islice(loader, max_batches)

    with torch.set_grad_enabled(training):
        for images, masks in iterator:
            images = images.to(device)
            masks = masks.to(device)
            if training:
                optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = criterion(logits, masks)
            if not torch.isfinite(loss).item():
                raise FloatingPointError("Non-finite Cross-Entropy loss.")
       

In [6]:
# Re-running this cell starts a NEW seeded experiment with fresh model/optimizer.
seed_everything(CONFIG["seed"])
loader_generator = torch.Generator().manual_seed(CONFIG["seed"])
loader_options = dict(
    batch_size=CONFIG["batch_size"],
    num_workers=CONFIG["num_workers"],
    pin_memory=DEVICE.type == "cuda",
    worker_init_fn=seed_worker,
)
train_loader = DataLoader(
    train_dataset, shuffle=True, generator=loader_generator, **loader_options
)
val_loader = DataLoader(val_dataset, shuffle=False, **loader_options)

model = SmallFCN().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["learning_rate"])
run_name = (
    f"{RUN_MODE}_{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}_{uuid4().hex[:8]}"
)
RUN_DIR = REPO_ROOT / "results/e0_small_fcn" / run_name

def git_output(*args):
    return subprocess.check_output(
        ["git", "-C", str(REPO_ROOT), *args], text=True
    ).strip()

metadata = {
    "python": platform.python_version(),
    "torch": str(torch.__version__),
    "torchvision": str(torchvision.__version__),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "pillow": PIL.__version__,
    "device": str(DEVICE),
    "device_name": torch.cuda.get_device_name() if DEVICE.type == "cuda" else platform.machine(),
    "cuda": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version(),
    "split_sha256": hashlib.sha256(SPLIT_CSV.read_bytes()).hexdigest(),
    "git_commit": git_output("rev-parse", "HEAD"),
    "git_tracked_changes": bool(git_output("status", "--porcelain", "--untracked-files=no")),
    "train_samples": len(train_dataset),
    "val_samples": len(val_dataset),
    "class_names": list(CLASS_NAMES),
    "trainable_parameters": count_trainable_parameters(model),
    "source_sha256": {
        str(p.relative_to(REPO_ROOT)): hashlib.sha256(p.read_bytes()).hexdigest()
        for p in [
            REPO_ROOT / "src/dataset.py", REPO_ROOT / "src/metrics.py",
            REPO_ROOT / "src/training.py", REPO_ROOT / "src/models/small_fcn.py",
        ]
    },
}
history = fit(
    model, train_loader, val_loader, optimizer, DEVICE, CONFIG, RUN_DIR, metadata
)
print("Saved run:", RUN_DIR.relative_to(REPO_ROOT))


[smoke] epoch 1/1 | train CE=1.1015 | val CE=1.0512 | val mIoU=0.1924


Saved run: results/e0_small_fcn/smoke_20261027T170049Z_5e9fae4c


## 5. Validation results

Load `best.pt` and evaluate it again to check that the saved model gives the recorded results. The table shows IoU for each class.

The configuration and environment are saved in `run.json`; per-epoch metrics are saved in `history.json`.


In [7]:
from src.training import run_epoch

checkpoint = torch.load(RUN_DIR / "best.pt", map_location=DEVICE, weights_only=True)
best_model = SmallFCN().to(DEVICE)
best_model.load_state_dict(checkpoint["model_state_dict"])
best_metrics = run_epoch(
    best_model, val_loader, DEVICE, max_batches=CONFIG["max_val_batches"]
)
assert best_metrics["confusion_matrix"] == checkpoint["val_metrics"]["confusion_matrix"]
assert abs(best_metrics["loss"] - checkpoint["val_metrics"]["loss"]) < 1e-6
print("Checkpoint round trip: passed")
print("Mode:", RUN_MODE, "| selected epoch:", checkpoint["epoch"])
print("Validation images evaluated:", best_metrics["samples"])
print(f"Validation mIoU: {best_metrics['miou']:.6f}")
display(pd.DataFrame({
    "class_id": range(3),
    "class": CLASS_NAMES,
    "IoU": best_metrics["per_class_iou"],
}))
if RUN_MODE == "smoke":
    print("SMOKE TEST ONLY: these numbers are not full-training baseline results.")


Checkpoint round trip: passed
Mode: smoke | selected epoch: 1
Validation images evaluated: 16
Validation mIoU: 0.192442


,class_id,class,IoU
0,0,foreground,0.000000
1,1,background,0.577327
2,2,border,0.000000


SMOKE TEST ONLY: these numbers are not full-training baseline results.


## 6. Curves and predictions

Plot the training and validation loss, validation mIoU, and one validation example with its predicted mask. The figures are saved as `curves.png` and `prediction.png` in the run folder.

Run this cell to display the figures.


In [8]:
epochs = [row["epoch"] for row in history]
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(epochs, [row["train"]["loss"] for row in history], "o-", label="train CE")
axes[0].plot(epochs, [row["val"]["loss"] for row in history], "o-", label="val CE")
axes[0].set(xlabel="Epoch", ylabel="Cross-Entropy", title=f"E0 — {RUN_MODE}")
axes[0].legend()
axes[1].plot(epochs, [row["val"]["miou"] for row in history], "o-")
axes[1].set(xlabel="Epoch", ylabel="Validation mIoU", ylim=(0, 1))
fig.tight_layout()
fig.savefig(RUN_DIR / "curves.png", dpi=140)
plt.show()
plt.close(fig)

from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.patches import Patch

image, target = val_dataset[0]
best_model.eval()
with torch.no_grad():
    prediction = best_model(image.unsqueeze(0).to(DEVICE)).argmax(1)[0].cpu()
colors = ["#d95f02", "#1b9e77", "#7570b3"]
cmap = ListedColormap(colors)
norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5], 3)
fig, axes = plt.subplots(1, 3, figsize=(10, 4))
axes[0].imshow((image.permute(1, 2, 0) * 0.5 + 0.5).clamp(0, 1))
axes[0].set_title("Validation image")
axes[1].imshow(target, cmap=cmap, norm=norm)
axes[1].set_title("Ground truth")
axes[2].imshow(prediction, cmap=cmap, norm=norm)
axes[2].set_title(f"Prediction — {RUN_MODE}")
for axis in axes:
    axis.axis("off")
fig.legend(
    handles=[Patch(color=color, label=name) for color, name in zip(colors, CLASS_NAMES)],
    loc="lower center", ncol=3,
)
fig.tight_layout(rect=(0, 0.08, 1, 0.96))
fig.savefig(RUN_DIR / "prediction.png", dpi=140)
plt.show()
plt.close(fig)


## 7. Run in Colab and save the results

Select a GPU runtime. Set `RUN_MODE = "full"` to train for the configured 20 epochs, or use `"smoke"` to check two batches per split for one epoch. Set `DOWNLOAD_DATA = True` if needed, then run all cells.

Each run gets a new folder under `results/e0_small_fcn/`. After training, set `DOWNLOAD_RUN = True` in the cell below to download the metrics, figures, and best checkpoint together.

Keep a copy of the ZIP before ending the Colab session. Checkpoints stay outside Git.


In [9]:
# Optional: download all artifacts together after training in Colab.
DOWNLOAD_RUN = False  # set True and rerun this cell when ready
if IN_COLAB and DOWNLOAD_RUN:
    import shutil
    from google.colab import files
    archive = shutil.make_archive(str(RUN_DIR), "zip", root_dir=RUN_DIR)
    files.download(archive)
